In [10]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    make_scorer,
)
import warnings
import os
warnings.filterwarnings("ignore")

SEED = 1312
np.random.seed(SEED)

# Paths
RAW_DATA_PATH = Path("../data/raw/telco_churn.csv")
MLFLOW_TRACKING_URI = os.path.abspath("../models/mlruns")

print("Imports OK")

Imports OK


In [11]:
df = pd.read_csv(RAW_DATA_PATH)

# Corrigir TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remover customerID e registros com TotalCharges nulo
df = df.drop(columns=["customerID"])
df = df.dropna(subset=["TotalCharges"])

# Separar features e target
X = df.drop(columns=["Churn"])
y = (df["Churn"] == "Yes").astype(int)

print(f"Shape X: {X.shape}")
print(f"Distribuição do target:\n{y.value_counts(normalize=True).round(3)}")

Shape X: (7032, 19)
Distribuição do target:
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64


In [12]:
# Separar colunas por tipo
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

print(f"Numéricas: {num_cols}")
print(f"Categóricas: {cat_cols}")

# Pipeline de pré-processamento
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ]
)

Numéricas: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categóricas: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [ ]:
# Configurar MLflow
import mlflow
from pathlib import Path

# SQLite como backend 
DB_PATH = Path("../models/mlflow.db").resolve()
MLFLOW_TRACKING_URI = f"sqlite:///{DB_PATH}"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("churn-baselines")

print("Tracking URI:", MLFLOW_TRACKING_URI)

# Validação cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Métricas
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": make_scorer(average_precision_score),
    "f1": make_scorer(f1_score),
    "recall": make_scorer(recall_score),
}

print("MLflow configurado em:", MLFLOW_TRACKING_URI)
print("Experiment: churn-baselines")

2026/04/23 23:28:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/23 23:28:34 INFO mlflow.store.db.utils: Updating database tables
2026/04/23 23:28:36 INFO mlflow.tracking.fluent: Experiment with name 'churn-baselines' does not exist. Creating a new experiment.


Tracking URI: sqlite:///C:\Users\bruno\OneDrive\Estudos\Especialização\Postech_MLE\modulo_01\Projeto-Churn-TC01\models\mlflow.db
MLflow configurado em: sqlite:///C:\Users\bruno\OneDrive\Estudos\Especialização\Postech_MLE\modulo_01\Projeto-Churn-TC01\models\mlflow.db
Experiment: churn-baselines


In [14]:
def train_and_log(name: str, classifier, X, y, cv, scoring, preprocessor):
    """Treina modelo com cross validation estratificada e registra no MLflow."""
    
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])
    
    with mlflow.start_run(run_name=name):
        # Registrar parâmetros
        mlflow.log_param("model", name)
        mlflow.log_param("cv_folds", cv.n_splits)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("n_samples", X.shape[0])
        mlflow.log_param("n_features", X.shape[1])
        
        # Validação cruzada
        results = cross_validate(
            pipeline, X, y,
            cv=cv,
            scoring=scoring,
            return_train_score=False,
        )
        
        # Registrar métricas médias
        metrics = {
            "roc_auc_mean": results["test_roc_auc"].mean(),
            "roc_auc_std": results["test_roc_auc"].std(),
            "pr_auc_mean": results["test_pr_auc"].mean(),
            "pr_auc_std": results["test_pr_auc"].std(),
            "f1_mean": results["test_f1"].mean(),
            "f1_std": results["test_f1"].std(),
            "recall_mean": results["test_recall"].mean(),
            "recall_std": results["test_recall"].std(),
        }
        mlflow.log_metrics(metrics)
        
        # Logar modelo
        pipeline.fit(X, y)
        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        
        print(f"\n{'='*45}")
        print(f"Modelo: {name}")
        print(f"  ROC-AUC : {metrics['roc_auc_mean']:.4f} ± {metrics['roc_auc_std']:.4f}")
        print(f"  PR-AUC  : {metrics['pr_auc_mean']:.4f} ± {metrics['pr_auc_std']:.4f}")
        print(f"  F1      : {metrics['f1_mean']:.4f} ± {metrics['f1_std']:.4f}")
        print(f"  Recall  : {metrics['recall_mean']:.4f} ± {metrics['recall_std']:.4f}")
        print(f"{'='*45}")
        
        return metrics

In [15]:
dummy_metrics = train_and_log(
    name="DummyClassifier",
    classifier=DummyClassifier(strategy="most_frequent", random_state=SEED),
    X=X, y=y, cv=cv, scoring=scoring, preprocessor=preprocessor,
)

2026/04/23 23:28:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 23:28:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/23 23:28:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Modelo: DummyClassifier
  ROC-AUC : 0.5000 ± 0.0000
  PR-AUC  : 0.2658 ± 0.0003
  F1      : 0.0000 ± 0.0000
  Recall  : 0.0000 ± 0.0000


In [16]:
lr_metrics = train_and_log(
    name="LogisticRegression",
    classifier=LogisticRegression(max_iter=1000, random_state=SEED),
    X=X, y=y, cv=cv, scoring=scoring, preprocessor=preprocessor,
)

2026/04/23 23:28:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 23:28:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/23 23:28:42 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Modelo: LogisticRegression
  ROC-AUC : 0.8450 ± 0.0080
  PR-AUC  : 0.4835 ± 0.0154
  F1      : 0.6023 ± 0.0175
  Recall  : 0.5570 ± 0.0254


In [17]:
results_df = pd.DataFrame({
    "Modelo": ["DummyClassifier", "LogisticRegression"],
    "ROC-AUC": [
        f"{dummy_metrics['roc_auc_mean']:.4f} ± {dummy_metrics['roc_auc_std']:.4f}",
        f"{lr_metrics['roc_auc_mean']:.4f} ± {lr_metrics['roc_auc_std']:.4f}",
    ],
    "PR-AUC": [
        f"{dummy_metrics['pr_auc_mean']:.4f} ± {dummy_metrics['pr_auc_std']:.4f}",
        f"{lr_metrics['pr_auc_mean']:.4f} ± {lr_metrics['pr_auc_std']:.4f}",
    ],
    "F1": [
        f"{dummy_metrics['f1_mean']:.4f} ± {dummy_metrics['f1_std']:.4f}",
        f"{lr_metrics['f1_mean']:.4f} ± {lr_metrics['f1_std']:.4f}",
    ],
    "Recall": [
        f"{dummy_metrics['recall_mean']:.4f} ± {dummy_metrics['recall_std']:.4f}",
        f"{lr_metrics['recall_mean']:.4f} ± {lr_metrics['recall_std']:.4f}",
    ],
})

print("\nTabela Comparativa de Baselines")
print(results_df.to_string(index=False))


Tabela Comparativa de Baselines
            Modelo         ROC-AUC          PR-AUC              F1          Recall
   DummyClassifier 0.5000 ± 0.0000 0.2658 ± 0.0003 0.0000 ± 0.0000 0.0000 ± 0.0000
LogisticRegression 0.8450 ± 0.0080 0.4835 ± 0.0154 0.6023 ± 0.0175 0.5570 ± 0.0254


In [18]:
print("\nPara visualizar os experimentos no MLflow UI, execute no terminal:")
print(f"  uv run mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}")
print("  Acesse: http://localhost:5000")


Para visualizar os experimentos no MLflow UI, execute no terminal:
  uv run mlflow ui --backend-store-uri sqlite:///C:\Users\bruno\OneDrive\Estudos\Especialização\Postech_MLE\modulo_01\Projeto-Churn-TC01\models\mlflow.db
  Acesse: http://localhost:5000
